####Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"

sol_balance_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.phantom_sol_balance"
)

sol_transactions_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.phantom_sol_transactions"
)

print(
    "Bronze SOL balance rows:",
    sol_balance_bronze_df.count()
)

print(
    "Bronze SOL transaction rows:",
    sol_transactions_bronze_df.count()
)

Bronze SOL balance rows: 1
Bronze SOL transaction rows: 7


####Sol Balance

In [0]:
decimal_type = DecimalType(38, 18)

phantom_sol_balance_silver_df = (
    sol_balance_bronze_df
    .select(
        col("source_file_name")
        .alias("snapshot_id"),

        col("wallet_address")
        .alias("account_id"),

        upper(col("asset"))
        .alias("asset"),

        col("balance")
        .cast(decimal_type)
        .alias("free_balance"),

        lit(0)
        .cast(decimal_type)
        .alias("locked_balance"),

        upper(col("network"))
        .alias("network"),

        col("wallet_address"),

        lit("Phantom")
        .alias("exchange"),

        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",
        col("free_balance")
        + col("locked_balance")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "snapshot_id",
        "account_id",
        "asset"
    ])
)

print(
    "Clean Phantom SOL balance rows:",
    phantom_sol_balance_silver_df.count()
)

phantom_sol_balance_silver_df.printSchema()

display(phantom_sol_balance_silver_df)

Clean Phantom SOL balance rows: 1
root
 |-- snapshot_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- free_balance: decimal(38,18) (nullable = true)
 |-- locked_balance: decimal(38,18) (nullable = false)
 |-- network: string (nullable = true)
 |-- wallet_address: string (nullable = true)
 |-- exchange: string (nullable = false)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- total_balance: decimal(38,17) (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



snapshot_id,account_id,asset,free_balance,locked_balance,network,wallet_address,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at
sol_balance_20261008T143029_084070Z_6f31d726.json,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOL,0.025388000000000000,0E-18,SOLANA,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,Phantom,s3://l1l2bucket/data/raw/phantom/sol_balance/sol_balance_20261008T143029_084070Z_6f31d726.json,2026-10-09T15:30:43.839Z,0.02538800000000000,2026-10-10T12:25:19.606Z


In [0]:
(
    phantom_sol_balance_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.phantom_account_balances"
    )
)

####Sol transactions

In [0]:
display(sol_transactions_bronze_df)

signature transaction source_file_name _source_file wallet_address wallet_name network source_type ingested_at 5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm List(1782684012, List(750, 2688, null, 11000, List(), List(Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(1102058845764807, 20103757, 25388000, 335789966, 42706560, 1, 1), List(), List(1102059196963807, 20103757, 0, 9989966, 42706560, 1, 1), List(), List(), List(null, null)), 429536564, List(List(List(List(5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, true, transaction, true), List(HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, false, transaction, true), List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, false, transaction, true), List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, false, transaction, true), List(SysvarRecentB1ockHashes11111111111111111111, false, transaction, false), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(null, null, List(List(null, null, HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, SysvarRecentB1ockHashes11111111111111111111, null), advanceNonce), system, 11111111111111111111111111111111, 1), List(List(), 3QAwFKa3MJAs, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), FKuNA3, null, null, ComputeBudget111111111111111111111111111111, 1), List(null, null, List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, 25388000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1), List(null, null, List(List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, 325800000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1)), D1g2vyB99L3wVqCN13FejqaZV9U9n35Md6AEWYYiUmYa), List(5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm)), 2144, legacy) sol_transactions_20261008T143032_460677Z_a3664059.json s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7 Phantom SOLANA Phantom Transaction 2026-10-09T15:37:13.363Z 2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M List(1773409311, List(450, 1784, null, 80000, List(), List(Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(0, 106414822, 1, 1), List(), List(6494922, 99999900, 1, 1), List(), List(), List(null, null)), 406146780, List(List(List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, true, transaction, true), List(7q3KWGdYaVsHEUZvDoZcJ9zCFNV3UYDY7ibLJq3aW8Ub, false, transaction, true), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(List(), 3b1H8Rq1T3d1, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), LKoyXd, null, null, ComputeBudget111111111111111111111111111111, 1), List(null, null, List(List(7q3KWGdYaVsHEUZvDoZcJ9zCFNV3UYDY7ibLJq3aW8Ub, 6414922, null

In [0]:
# ========================================================
# FIND PHANTOM WALLET BALANCE IN EACH TRANSACTION
# ========================================================

phantom_wallet_rows_df = (
    sol_transactions_bronze_df
    .withColumn(
        "wallet_row",
        explode(
            arrays_zip(
                col(
                    "transaction.transaction.message.accountKeys"
                ).alias("account"),

                col(
                    "transaction.meta.preBalances"
                ).alias("pre_balance"),

                col(
                    "transaction.meta.postBalances"
                ).alias("post_balance")
            )
        )
    )
    .filter(
        col("wallet_row.account.pubkey")
        == col("wallet_address")
    )
)

print(
    "Matched Phantom transaction rows:",
    phantom_wallet_rows_df.count()
)

display(phantom_wallet_rows_df)

Matched Phantom transaction rows: 7


signature transaction source_file_name _source_file wallet_address wallet_name network source_type ingested_at wallet_row 5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm List(1782684012, List(750, 2688, null, 11000, List(), List(Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(1102058845764807, 20103757, 25388000, 335789966, 42706560, 1, 1), List(), List(1102059196963807, 20103757, 0, 9989966, 42706560, 1, 1), List(), List(), List(null, null)), 429536564, List(List(List(List(5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, true, transaction, true), List(HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, false, transaction, true), List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, false, transaction, true), List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, false, transaction, true), List(SysvarRecentB1ockHashes11111111111111111111, false, transaction, false), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(null, null, List(List(null, null, HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, SysvarRecentB1ockHashes11111111111111111111, null), advanceNonce), system, 11111111111111111111111111111111, 1), List(List(), 3QAwFKa3MJAs, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), FKuNA3, null, null, ComputeBudget111111111111111111111111111111, 1), List(null, null, List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, 25388000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1), List(null, null, List(List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, 325800000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1)), D1g2vyB99L3wVqCN13FejqaZV9U9n35Md6AEWYYiUmYa), List(5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm)), 2144, legacy) sol_transactions_20261008T143032_460677Z_a3664059.json s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7 Phantom SOLANA Phantom Transaction 2026-10-09T15:37:13.363Z List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, false, transaction, true), 0, 25388000) 2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M List(1773409311, List(450, 1784, null, 80000, List(), List(Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(0, 106414822, 1, 1), List(), List(6494922, 99999900, 1, 1), List(), List(), List(null, null)), 406146780, List(List(List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, true, transaction, true), List(7q3KWGdYaVsHEUZvDoZcJ9zCFNV3UYDY7ibLJq3aW8Ub, false, transaction, true), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(List(), 3b1H8Rq1T3d1, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), LKoyXd, null, null, ComputeBudget111111111111111

In [0]:
# ========================================================
# CLEAN PHANTOM SOL TRANSACTIONS
# ========================================================

phantom_sol_transactions_silver_df = (
    phantom_wallet_rows_df

    .withColumn(
        "balance_change_lamports",
        col("wallet_row.post_balance")
        - col("wallet_row.pre_balance")
    )

    .withColumn(
        "wallet_is_signer",
        col("wallet_row.account.signer")
    )

    .withColumn(
        "status",
        when(
            col("transaction.meta.err").isNotNull(),
            lit("FAILED")
        )
        .otherwise(
            lit("SUCCESS")
        )
    )

    .withColumn(
        "direction",
        when(
            col("status") == "FAILED",
            lit("FAILED")
        )
        .when(
            col("balance_change_lamports") > 0,
            lit("IN")
        )
        .when(
            col("balance_change_lamports") < 0,
            lit("OUT")
        )
        .otherwise(
            lit("NO_CHANGE")
        )
    )

    .withColumn(
        "wallet_fee_lamports",
        when(
            col("wallet_is_signer") == True,
            col("transaction.meta.fee")
        )
        .otherwise(
            lit(0)
        )
    )

    .withColumn(
        "transfer_lamports",
        when(
            col("status") == "FAILED",
            lit(0)
        )
        .when(
            col("direction") == "IN",
            col("balance_change_lamports")
        )
        .when(
            col("direction") == "OUT",
            abs(col("balance_change_lamports"))
            - col("wallet_fee_lamports")
        )
        .otherwise(
            lit(0)
        )
    )

    .select(
        col("signature")
        .alias("transaction_id"),

        lit("SOL")
        .alias("asset"),

        col("direction"),

        (
            col("transfer_lamports")
            / lit(1_000_000_000)
        )
        .cast("decimal(38,18)")
        .alias("amount"),

        (
            col("wallet_fee_lamports")
            / lit(1_000_000_000)
        )
        .cast("decimal(38,18)")
        .alias("transaction_fee"),

        (
            col("balance_change_lamports")
            / lit(1_000_000_000)
        )
        .cast("decimal(38,18)")
        .alias("balance_change"),

        col("status"),

        timestamp_seconds(
            col("transaction.blockTime")
        ).alias("transaction_time"),

        col("transaction.slot")
        .alias("slot"),

        col("wallet_address"),

        lit("SOLANA")
        .alias("network"),

        lit("Phantom")
        .alias("exchange"),

        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at"),

        current_timestamp()
        .alias("silver_processed_at")
    )

    .dropDuplicates([
        "transaction_id"
    ])
)

print(
    "Clean Phantom SOL transaction rows:",
    phantom_sol_transactions_silver_df.count()
)

display(
    phantom_sol_transactions_silver_df
    .orderBy(
        col("transaction_time")
    )
)

Clean Phantom SOL transaction rows: 7


transaction_id,asset,direction,amount,transaction_fee,balance_change,status,transaction_time,slot,wallet_address,network,exchange,_source_file,bronze_ingested_at,silver_processed_at
kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,SOL,IN,0.106654820000000000,0E-18,0.106654820000000000,SUCCESS,2026-03-13T06:39:30.000Z,406082110,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
4bW3DTm7hfmaEmpavEHc92XWkunhVJFr76J3rN592wi6H26wpkkbAQWxDKnTxgD8Ky53XCnsKocC223SUmezBVFf,SOL,FAILED,0E-18,0.000080000000000000,-0.000080000000000000,FAILED,2026-03-13T13:34:55.000Z,406145724,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
2eMTSsX64vspd93GeSffoL84BBo8pvfDwRNpD51446sHFZH3SEpD4rg3wJJa7euTh7qwUkRY33BJgbnwyujWBYv6,SOL,OUT,0.099999900000000000,0.000080000000000000,-0.100079900000000000,SUCCESS,2026-03-13T13:36:15.000Z,406145929,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
4B1MJE5DHiKKpzmezWYDaJG7EFrPU8WgW5GTqrNZXYxxKDQKk5XbjY5gzUTk6of364xLBYRUgwjyPCXsqVQxtA9m,SOL,IN,1.000000000E-9,0E-18,1.000000000E-9,SUCCESS,2026-03-13T13:36:24.000Z,406145953,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
3xjSrZJi8e4cyRP97uEZFPM1gQKmWXdPBNfigFFQUjRarYk5hFFd9CsCo8HDKcCzYFQYrC8iJDo4YhpbFF2w3Sjx,SOL,IN,1.000000000E-9,0E-18,1.000000000E-9,SUCCESS,2026-03-13T13:36:24.000Z,406145953,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M,SOL,OUT,0.006414922000000000,0.000080000000000000,-0.006494922000000000,SUCCESS,2026-03-13T13:41:51.000Z,406146780,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z
5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm,SOL,IN,0.025388000000000000,0E-18,0.025388000000000000,SUCCESS,2026-06-28T22:00:12.000Z,429536564,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,SOLANA,Phantom,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json,2026-10-09T15:37:13.363Z,2026-10-10T12:43:12.939Z


In [0]:
(
    phantom_sol_transactions_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.phantom_sol_transactions"
    )
)